In [ ]:
import random
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets

# Grid Dimensions
WIDTH = 20
HEIGHT = 15

# Game State Variables
snake = [[7, 10], [7, 9], [7, 8]]
direction = 'RIGHT'
fruit = [random.randint(1, HEIGHT-2), random.randint(1, WIDTH-2)]
score = 0
game_over = False

# Render the game frame with Modern Neon Styling
def draw_game():
    global game_over, score, fruit

    css_styles = """
    <style>
        .game-container {
            font-family: 'Courier New', monospace;
            line-height: 16px;
            letter-spacing: 3px;
            font-size: 20px;
            background: radial-gradient(circle, #1a1a2e 0%, #0f0c1b 100%);
            padding: 20px;
            border-radius: 15px;
            display: inline-block;
            border: 3px solid #00f2fe;
            box-shadow: 0 0 20px rgba(0, 242, 254, 0.3);
        }
        .score-board {
            color: #00f2fe;
            font-family: 'Segoe UI', sans-serif;
            font-size: 18px;
            margin-top: 15px;
            text-align: center;
            font-weight: bold;
            text-shadow: 0 0 10px rgba(0, 242, 254, 0.6);
        }
        .custom-btn {
            background: linear-gradient(135deg, #00f2fe 0%, #4facfe 100%) !important;
            color: white !important;
            font-weight: bold !important;
            border-radius: 50% !important;
            box-shadow: 0 4px 15px rgba(0, 242, 254, 0.4) !important;
            border: none !important;
        }
    </style>
    """

    html_grid = css_styles + "<div class='game-container'>"

    for r in range(HEIGHT):
        line = ""
        for c in range(WIDTH):
            if r == 0 or r == HEIGHT - 1 or c == 0 or c == WIDTH - 1:
                line += "<span style='color: #4facfe;'>█</span>"
            elif [r, c] == snake[0]:
                line += "<span style='color: #00ff87; font-weight: bold; text-shadow: 0 0 8px #00ff87;'>O</span>"
            elif [r, c] in snake[1:]:
                line += "<span style='color: #60efff;'>o</span>"
            elif [r, c] == fruit:
                line += "<span style='color: #ff007f; text-shadow: 0 0 10px #ff007f;'>■</span>"
            else:
                line += "<span style='color: #252542;'>.</span>"
        html_grid += line + "<br>"

    html_grid += f"<div class='score-board'>⚡ NEON SCORE: {score} ⚡</div>"
    html_grid += "</div>"

    clear_output(wait=True)
    display(HTML(html_grid))
    display(control_layout)

# Main engine logic called by JavaScript heartbeat timer
def game_tick(change=None):
    global direction, fruit, score, game_over, snake

    if game_over:
        return

    head = list(snake[0])
    if direction == 'UP': head[0] -= 1
    elif direction == 'DOWN': head[0] += 1
    elif direction == 'LEFT': head[1] -= 1
    elif direction == 'RIGHT': head[1] += 1

    # Collision Detection (Walls)
    if head[0] <= 0 or head[0] >= HEIGHT - 1 or head[1] <= 0 or head[1] >= WIDTH - 1:
        game_over = True
        end_game()
        return

    # Collision Detection (Self-bite)
    if head in snake:
        game_over = True
        end_game()
        return

    snake.insert(0, head)

    # Food mechanism
    if head == fruit:
        score += 10
        while True:
            fruit = [random.randint(1, HEIGHT-2), random.randint(1, WIDTH-2)]
            if fruit not in snake: break
    else:
        snake.pop()

    draw_game()

def end_game():
    clear_output(wait=True)
    end_html = f"""
    <div style='text-align: center; font-family: sans-serif; background: #0f0c1b; padding: 40px; border-radius: 15px; border: 3px solid #ff007f; display: inline-block; box-shadow: 0 0 20px rgba(255, 0, 127, 0.4);'>
        <h1 style='color: #ff007f; margin: 0; text-shadow: 0 0 10px #ff007f;'>GAME OVER</h1>
        <p style='color: #ffffff; font-size: 18px; margin: 15px 0;'>Your Final Score: <strong style='color: #00f2fe;'>{score}</strong></p>
        <p style='color: #666;'>Re-run the Colab cell to play again!</p>
    </div>
    """
    display(HTML(end_html))

# Control logic handlers
def on_up_click(b):
    global direction
    if direction != 'DOWN': direction = 'UP'

def on_down_click(b):
    global direction
    if direction != 'UP': direction = 'DOWN'

def on_left_click(b):
    global direction
    if direction != 'RIGHT': direction = 'LEFT'

def on_right_click(b):
    global direction
    if direction != 'LEFT': direction = 'RIGHT'

# Widgets UI Initialization
btn_up = widgets.Button(description='▲', layout=widgets.Layout(width='50px', height='50px'))
btn_down = widgets.Button(description='▼', layout=widgets.Layout(width='50px', height='50px'))
btn_left = widgets.Button(description='◀', layout=widgets.Layout(width='50px', height='50px'))
btn_right = widgets.Button(description='▶', layout=widgets.Layout(width='50px', height='50px'))

btn_up.add_class('custom-btn')
btn_down.add_class('custom-btn')
btn_left.add_class('custom-btn')
btn_right.add_class('custom-btn')

btn_up.on_click(on_up_click)
btn_down.on_click(on_down_click)
btn_left.on_click(on_left_click)
btn_right.on_click(on_right_click)

top_row = widgets.HBox([btn_up], layout=widgets.Layout(justify_content='center'))
middle_row = widgets.HBox([btn_left, btn_right], layout=widgets.Layout(justify_content='center', gap='25px'))
bottom_row = widgets.HBox([btn_down], layout=widgets.Layout(justify_content='center'))
control_layout = widgets.VBox([top_row, middle_row, bottom_row], layout=widgets.Layout(margin='20px 0 0 0'))

# Hidden Play Button used as an internal bridge to catch JavaScript interval pulses
js_bridge = widgets.Play(interval=200, value=0, min=0, max=100000, show_repeat=False, autoplay=True)
js_bridge.observe(game_tick, 'value')

# Boot game layout
draw_game()
display(js_bridge) # Running invisible JS intervals inside backend container

Play(value=0, interval=200, max=100000, show_repeat=False)

In [ ]:
from IPython.display import display, HTML

# 100% Client-Side Pure HTML5/JavaScript Snake Game
# This runs entirely in your browser, bypassing Colab's backend limitations to prevent freezes/errors.

game_code = """
<div id="canvas-container" style="text-align: center; background: radial-gradient(circle, #121224 0%, #080711 100%); padding: 25px; border-radius: 20px; border: 4px solid #00f2fe; box-shadow: 0 0 25px rgba(0, 242, 254, 0.4); display: inline-block; font-family: 'Segoe UI', sans-serif;">

    <div style="color: #00f2fe; font-size: 26px; font-weight: bold; margin-bottom: 10px; text-shadow: 0 0 10px #00f2fe;">
        ⚡ NEON ARCADE SNAKE ⚡
    </div>

    <div id="score-text" style="color: #ff007f; font-size: 20px; font-weight: bold; margin-bottom: 15px; text-shadow: 0 0 8px #ff007f;">
        SCORE: 0
    </div>

    <canvas id="gameCanvas" width="360" height="280" style="background-color: #05050f; border: 2px solid #4facfe; border-radius: 10px; box-shadow: inset 0 0 15px rgba(79, 172, 254, 0.3);"></canvas>

    <div style="margin-top: 20px; display: grid; grid-template-columns: repeat(3, 65px); grid-gap: 12px; justify-content: center;">
        <div></div>
        <button class="control-btn" onclick="setDir('UP')">▲</button>
        <div></div>
        <button class="control-btn" onclick="setDir('LEFT')">◀</button>
        <button class="restart-btn" onclick="restart()">🔄</button>
        <button class="control-btn" onclick="setDir('RIGHT')">▶</button>
        <div></div>
        <button class="control-btn" onclick="setDir('DOWN')">▼</button>
        <div></div>
    </div>
</div>

<style>
    /* Styling the circular control buttons */
    .control-btn {
        width: 65px;
        height: 65px;
        background: linear-gradient(135deg, #00f2fe 0%, #4facfe 100%);
        color: white;
        border: none;
        border-radius: 50%;
        font-size: 24px;
        font-weight: bold;
        cursor: pointer;
        box-shadow: 0 5px 15px rgba(0, 242, 254, 0.4);
        transition: transform 0.1s ease, box-shadow 0.1s ease;
    }
    .control-btn:active {
        transform: scale(0.9);
        box-shadow: 0 2px 5px rgba(0, 242, 254, 0.6);
    }
    /* Styling the circular restart button */
    .restart-btn {
        width: 65px;
        height: 65px;
        background: linear-gradient(135deg, #ff007f 0%, #7928ca 100%);
        color: white;
        border: none;
        border-radius: 50%;
        font-size: 22px;
        cursor: pointer;
        box-shadow: 0 5px 15px rgba(255, 0, 127, 0.4);
        transition: transform 0.1s ease;
    }
    .restart-btn:active {
        transform: scale(0.9);
    }
</style>

<script>
    const canvas = document.getElementById("gameCanvas");
    const ctx = canvas.getContext("2d");
    const scoreText = document.getElementById("score-text");

    const size = 20; // Size of each block
    const rows = canvas.height / size;
    const cols = canvas.width / size;

    let snake, dir, food, score, isOver, loop;

    // Start or Restart the game
    function start() {
        snake = [{x: 8, y: 6}, {x: 7, y: 6}, {x: 6, y: 6}];
        dir = "RIGHT";
        score = 0;
        isOver = false;
        scoreText.innerText = "SCORE: " + score;
        spawnFood();

        if (loop) clearInterval(loop);
        // Delay set to 260ms for a perfectly slow and easy speed
        loop = setInterval(update, 260);
    }

    // Spawn food on a random empty tile
    function spawnFood() {
        food = {
            x: Math.floor(Math.random() * cols),
            y: Math.floor(Math.random() * rows)
        };
        // Check if food spawned on top of the snake body
        for (let cell of snake) {
            if (cell.x === food.x && cell.y === food.y) {
                spawnFood();
                break;
            }
        }
    }

    // Update direction safely
    function setDir(newDir) {
        if (newDir === "UP" && dir !== "DOWN") dir = "UP";
        if (newDir === "DOWN" && dir !== "UP") dir = "DOWN";
        if (newDir === "LEFT" && dir !== "RIGHT") dir = "LEFT";
        if (newDir === "RIGHT" && dir !== "LEFT") dir = "RIGHT";
    }

    function restart() {
        start();
    }

    // Game logic tick
    function update() {
        if (isOver) return;

        let head = {x: snake[0].x, y: snake[0].y};
        if (dir === "UP") head.y--;
        if (dir === "DOWN") head.y++;
        if (dir === "LEFT") head.x--;
        if (dir === "RIGHT") head.x++;

        // Wall collisions
        if (head.x < 0 || head.x >= cols || head.y < 0 || head.y >= rows) {
            gameOver();
            return;
        }

        // Self collisions
        for (let cell of snake) {
            if (head.x === cell.x && head.y === cell.y) {
                gameOver();
                return;
            }
        }

        snake.unshift(head);

        // Eating food check
        if (head.x === food.x && head.y === food.y) {
            score += 10;
            scoreText.innerText = "SCORE: " + score;
            spawnFood();
        } else {
            snake.pop();
        }

        render();
    }

    // Render elements on the canvas
    function render() {
        // Clear screen
        ctx.fillStyle = "#05050f";
        ctx.fillRect(0, 0, canvas.width, canvas.height);

        // Draw Food with a beautiful neon-pink glow
        ctx.fillStyle = "#ff007f";
        ctx.shadowBlur = 10;
        ctx.shadowColor = "#ff007f";
        ctx.fillRect(food.x * size + 2, food.y * size + 2, size - 4, size - 4);

        // Draw Snake with a glowing gradient neon-green look
        snake.forEach((cell, i) => {
            ctx.fillStyle = i === 0 ? "#00ff87" : "#00f2fe";
            ctx.shadowBlur = i === 0 ? 12 : 6;
            ctx.shadowColor = i === 0 ? "#00ff87" : "#00f2fe";
            ctx.fillRect(cell.x * size + 1, cell.y * size + 1, size - 2, size - 2);
        });

        ctx.shadowBlur = 0; // Reset shadows for performance
    }

    // Handle game over state
    function gameOver() {
        isOver = true;
        clearInterval(loop);

        ctx.fillStyle = "rgba(8, 7, 17, 0.85)";
        ctx.fillRect(0, 0, canvas.width, canvas.height);

        ctx.fillStyle = "#ff007f";
        ctx.font = "bold 26px 'Segoe UI', sans-serif";
        ctx.textAlign = "center";
        ctx.shadowBlur = 12;
        ctx.shadowColor = "#ff007f";
        ctx.fillText("GAME OVER", canvas.width / 2, canvas.height / 2 - 10);

        ctx.fillStyle = "#ffffff";
        ctx.font = "14px sans-serif";
        ctx.shadowBlur = 0;
        ctx.fillText("Click 🔄 to play again", canvas.width / 2, canvas.height / 2 + 25);
    }

    // Add support for WASD and Arrow Keys on desktop/notebook
    document.addEventListener("keydown", function(e) {
        if (e.key === "ArrowUp" || e.key === "w" || e.key === "W") setDir("UP");
        if (e.key === "ArrowDown" || e.key === "s" || e.key === "S") setDir("DOWN");
        if (e.key === "ArrowLeft" || e.key === "a" || e.key === "A") setDir("LEFT");
        if (e.key === "ArrowRight" || e.key === "d" || e.key === "D") setDir("RIGHT");
    });

    // Initialize the game on load
    start();
</script>
"""

display(HTML(game_code))

In [ ]:
from google.colab import files

# Creating a single self-contained HTML file with Polygon button and smooth gameplay
html_file_content = """<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Neon Arcade Snake</title>
    <style>
        body {
            margin: 0;
            padding: 0;
            background: radial-gradient(circle, #121224 0%, #080711 100%);
            display: flex;
            justify-content: center;
            align-items: center;
            height: 100vh;
            font-family: 'Segoe UI', sans-serif;
            overflow: hidden;
        }

        #game-wrapper {
            text-align: center;
            background: rgba(20, 20, 40, 0.6);
            padding: 30px;
            border-radius: 20px;
            border: 4px solid #00f2fe;
            box-shadow: 0 0 30px rgba(0, 242, 254, 0.4);
            position: relative;
        }

        #score-text {
            color: #ff007f;
            font-size: 24px;
            font-weight: bold;
            margin-bottom: 15px;
            text-shadow: 0 0 10px #ff007f;
        }

        canvas {
            background-color: #05050f;
            border: 2px solid #4facfe;
            border-radius: 10px;
            box-shadow: inset 0 0 25px rgba(79, 172, 254, 0.3);
            display: none; /* Hidden until start */
        }

        /* Polygon (Hexagon) Menu Overlay */
        #menu-overlay {
            width: 400px;
            height: 300px;
            display: flex;
            justify-content: center;
            align-items: center;
            background: #05050f;
            border: 2px solid #4facfe;
            border-radius: 10px;
        }

        /* Unique Glowing Polygon (Hexagon) Button Shape */
        .polygon-btn {
            width: 160px;
            height: 140px;
            background: linear-gradient(135deg, #00f2fe 0%, #ff007f 100%);
            color: white;
            font-size: 18px;
            font-weight: bold;
            border: none;
            cursor: pointer;
            /* Creating the Hexagon Shape using CSS Clip-Path */
            clip-path: polygon(50% 0%, 100% 25%, 100% 75%, 50% 100%, 0% 75%, 0% 25%);
            display: flex;
            justify-content: center;
            align-items: center;
            text-align: center;
            box-shadow: 0 0 20px rgba(0, 242, 254, 0.6);
            transition: transform 0.2s, filter 0.2s;
        }

        .polygon-btn:hover {
            transform: scale(1.05);
            filter: drop-shadow(0 0 15px #00f2fe);
        }

        .polygon-btn:active {
            transform: scale(0.95);
        }

        /* Circular Control Pad Grid */
        #controls {
            margin-top: 25px;
            display: grid;
            grid-template-columns: repeat(3, 65px);
            grid-gap: 12px;
            justify-content: center;
        }

        .control-btn {
            width: 65px;
            height: 65px;
            background: linear-gradient(135deg, #00f2fe 0%, #4facfe 100%);
            color: white;
            border: none;
            border-radius: 50%;
            font-size: 24px;
            font-weight: bold;
            cursor: pointer;
            box-shadow: 0 5px 15px rgba(0, 242, 254, 0.3);
            transition: transform 0.1s, box-shadow 0.1s;
        }

        .control-btn:active {
            transform: scale(0.9);
            box-shadow: 0 2px 5px rgba(0, 242, 254, 0.5);
        }
    </style>
</head>
<body>

<div id="game-wrapper">
    <div id="score-text">⚡ NEON SNAKE ⚡</div>

    <div id="menu-overlay">
        <button class="polygon-btn" onclick="startGame()">START<br>GAME</button>
    </div>

    <canvas id="gameCanvas" width="400" height="300"></canvas>

    <div id="controls">
        <div></div>
        <button class="control-btn" onclick="setDir('UP')">▲</button>
        <div></div>
        <button class="control-btn" onclick="setDir('LEFT')">◀</button>
        <div></div>
        <button class="control-btn" onclick="setDir('RIGHT')">▶</button>
        <div></div>
        <button class="control-btn" onclick="setDir('DOWN')">▼</button>
        <div></div>
    </div>
</div>

<script>
    const canvas = document.getElementById("gameCanvas");
    const ctx = canvas.getContext("2d");
    const scoreText = document.getElementById("score-text");
    const menuOverlay = document.getElementById("menu-overlay");

    const size = 20;
    const rows = canvas.height / size;
    const cols = canvas.width / size;

    let snake, dir, food, score, isOver, loop;

    function startGame() {
        // Hide the polygon menu and show the canvas game view
        menuOverlay.style.display = "none";
        canvas.style.display = "block";

        // Reset States
        snake = [{x: 9, y: 7}, {x: 8, y: 7}, {x: 7, y: 7}];
        dir = "RIGHT";
        score = 0;
        isOver = false;
        scoreText.innerText = "SCORE: " + score;
        spawnFood();

        if (loop) clearInterval(loop);
        // Slower 280ms ticker speed for completely relaxing easy control vectors
        loop = setInterval(update, 280);
    }

    function spawnFood() {
        food = {
            x: Math.floor(Math.random() * cols),
            y: Math.floor(Math.random() * rows)
        };
        for (let cell of snake) {
            if (cell.x === food.x && cell.y === food.y) {
                spawnFood();
                break;
            }
        }
    }

    function setDir(newDir) {
        if (newDir === "UP" && dir !== "DOWN") dir = "UP";
        if (newDir === "DOWN" && dir !== "UP") dir = "DOWN";
        if (newDir === "LEFT" && dir !== "RIGHT") dir = "LEFT";
        if (newDir === "RIGHT" && dir !== "LEFT") dir = "RIGHT";
    }

    function update() {
        if (isOver) return;

        let head = {x: snake[0].x, y: snake[0].y};
        if (dir === "UP") head.y--;
        if (dir === "DOWN") head.y++;
        if (dir === "LEFT") head.x--;
        if (dir === "RIGHT") head.x++;

        // Edge Collision Detection
        if (head.x < 0 || head.x >= cols || head.y < 0 || head.y >= rows) {
            gameOver();
            return;
        }

        // Self-collision Detection
        for (let cell of snake) {
            if (head.x === cell.x && head.y === cell.y) {
                gameOver();
                return;
            }
        }

        snake.unshift(head);

        if (head.x === food.x && head.y === food.y) {
            score += 10;
            scoreText.innerText = "SCORE: " + score;
            spawnFood();
        } else {
            snake.pop();
        }

        render();
    }

    function render() {
        ctx.fillStyle = "#05050f";
        ctx.fillRect(0, 0, canvas.width, canvas.height);

        // Neon Pink Target Target Fruit
        ctx.fillStyle = "#ff007f";
        ctx.shadowBlur = 10;
        ctx.shadowColor = "#ff007f";
        ctx.fillRect(food.x * size + 2, food.y * size + 2, size - 4, size - 4);

        // Neon Green/Blue Gradient Snake Segments
        snake.forEach((cell, i) => {
            ctx.fillStyle = i === 0 ? "#00ff87" : "#00f2fe";
            ctx.shadowBlur = i === 0 ? 12 : 6;
            ctx.shadowColor = i === 0 ? "#00ff87" : "#00f2fe";
            ctx.fillRect(cell.x * size + 1, cell.y * size + 1, size - 2, size - 2);
        });

        ctx.shadowBlur = 0;
    }

    function gameOver() {
        isOver = true;
        clearInterval(loop);

        // Show the menu overlay screen container setup again
        canvas.style.display = "none";
        menuOverlay.style.display = "flex";

        // Re-label the polygon action button description
        const polyBtn = document.querySelector(".polygon-btn");
        polyBtn.innerHTML = "RESTART<br>GAME";
        scoreText.innerText = "GAME OVER! SCORE: " + score;
    }

    // Connect standard keyboard listeners
    document.addEventListener("keydown", function(e) {
        if (e.key === "ArrowUp" || e.key === "w" || e.key === "W") setDir("UP");
        if (e.key === "ArrowDown" || e.key === "s" || e.key === "S") setDir("DOWN");
        if (e.key === "ArrowLeft" || e.key === "a" || e.key === "A") setDir("LEFT");
        if (e.key === "ArrowRight" || e.key === "d" || e.key === "D") setDir("RIGHT");
    });
</script>

</body>
</html>
"""

# Saving the data string into a local file block context
with open("neon_snake.html", "w", encoding="utf-8") as f:
    f.write(html_file_content)

# Instantly download the script artifact inside browser downloads directory
files.download("neon_snake.html")
print("Successfully generated and downloaded 'neon_snake.html'! Double click the file locally to play!")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Successfully generated and downloaded 'neon_snake.html'! Double click the file locally to play!
